In [2]:
from pathlib import Path
import time
import duckdb
import numpy as np
import pandas as pd
import lightgbm as lgb
from metrics import apk

print("lightgbm:", lgb.__version__)

DATA = Path("data")
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE VIEW tx AS SELECT * FROM '{(DATA / 'sample_transactions.parquet').as_posix()}'")
con.execute(f"CREATE VIEW customers AS SELECT * FROM '{(DATA / 'customers.parquet').as_posix()}'")

# Text embeddings from Step 8
emb = np.load(DATA / "article_emb.npy")
emb_ids = np.load(DATA / "article_emb_ids.npy")
row_of = {a: i for i, a in enumerate(emb_ids)}

# Validation answer key
labels = con.execute(f"SELECT customer_id, bought FROM '{(DATA / 'labels_valid.parquet').as_posix()}'").df()
actual = {c: list(b) for c, b in zip(labels["customer_id"], labels["bought"])}

print(f"Embeddings: {emb.shape} | validation customers: {len(actual):,}")

lightgbm: 4.7.0
Embeddings: (105542, 384) | validation customers: 14,471


In [3]:
def candidates(t, custs):
    """Candidates for target week t, using only weeks t+1 .. t+8."""
    lo, hi = t + 1, t + 8
    cdf = pd.DataFrame({"customer_id": custs})
    con.register("cust_t", cdf)

    past = con.execute(f"""
        SELECT tx.customer_id, article_id FROM tx JOIN cust_t USING (customer_id)
        WHERE w BETWEEN {lo} AND {hi} GROUP BY ALL""").df()

    partners = con.execute(f"""
        WITH b AS (SELECT DISTINCT customer_id, t_dat, article_id FROM tx WHERE w BETWEEN {lo} AND {hi}),
        p AS (SELECT x.article_id AS item, y.article_id AS partner, COUNT(*) AS n
              FROM b x JOIN b y ON x.customer_id = y.customer_id AND x.t_dat = y.t_dat
                                AND x.article_id <> y.article_id
              GROUP BY ALL),
        r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY item ORDER BY n DESC, partner) AS rk FROM p)
        SELECT item, partner FROM r WHERE rk <= 3""").df()
    part = (past.merge(partners, left_on="article_id", right_on="item")
                [["customer_id", "partner"]].rename(columns={"partner": "article_id"}))

    pop = con.execute(f"""
        SELECT article_id FROM tx WHERE w BETWEEN {lo} AND {lo + 1}
        GROUP BY article_id ORDER BY COUNT(DISTINCT customer_id) DESC, article_id LIMIT 30""").df()
    popc = cdf.merge(pop, how="cross")

    pool = con.execute(f"SELECT DISTINCT article_id FROM tx WHERE w BETWEEN {lo} AND {lo + 1}").df()["article_id"].to_numpy()
    pool_emb = emb[[row_of[a] for a in pool]]
    grp = past.groupby("customer_id")["article_id"].apply(list)
    sims, cs = [], grp.index.tolist()
    for s in range(0, len(cs), 2000):
        batch = cs[s:s + 2000]
        prof = np.stack([emb[[row_of[a] for a in grp[c]]].mean(axis=0) for c in batch])
        prof /= np.linalg.norm(prof, axis=1, keepdims=True)
        top = np.argpartition(-(prof @ pool_emb.T), 10, axis=1)[:, :10]
        sims += [(c, pool[j]) for k, c in enumerate(batch) for j in top[k]]
    simc = pd.DataFrame(sims, columns=["customer_id", "article_id"])

    return pd.concat([past, part, popc, simc]).drop_duplicates().reset_index(drop=True)

t0 = time.time()
cand_v = candidates(1, list(actual))
print(f"Validation candidates: {len(cand_v):,} rows | {len(cand_v)/len(actual):.1f} per customer | {time.time()-t0:.0f}s")

pos_v = labels.explode("bought").rename(columns={"bought": "article_id"})
hit = pos_v.merge(cand_v, on=["customer_id", "article_id"])
print(f"Recall: {len(hit)/len(pos_v):.2%} of real purchases are in the candidates")
print(f"Customers with at least one bought item in candidates: {hit['customer_id'].nunique()/len(actual):.2%}")

Validation candidates: 726,718 rows | 50.2 per customer | 3s
Recall: 7.72% of real purchases are in the candidates
Customers with at least one bought item in candidates: 19.18%


In [4]:
def features(cand, t):
    lo, hi = t + 1, t + 8
    ref = (pd.Timestamp("2020-09-22") - pd.Timedelta(days=7 * t + 7)).date()  # last history day
    con.register("cand", cand)
    df = con.execute(f"""
        WITH h AS (SELECT * FROM tx WHERE w BETWEEN {lo} AND {hi}),
        ci AS (SELECT customer_id, article_id, COUNT(*) AS n_buy,
                      DATE_DIFF('day', MAX(t_dat), DATE '{ref}') AS days_since
               FROM h GROUP BY ALL),
        it AS (SELECT article_id,
                      COUNT(DISTINCT CASE WHEN w = {lo} THEN customer_id END) AS pop_1w,
                      COUNT(DISTINCT CASE WHEN w <= {lo + 1} THEN customer_id END) AS pop_2w,
                      COUNT(DISTINCT customer_id) AS pop_8w,
                      AVG(price) AS item_price
               FROM h GROUP BY article_id),
        age_i AS (SELECT article_id, MAX(w) - {lo} AS item_age_w FROM tx WHERE w >= {lo} GROUP BY article_id),
        cu AS (SELECT customer_id, COUNT(*) AS cust_n, AVG(price) AS cust_price,
                      DATE_DIFF('day', MAX(t_dat), DATE '{ref}') AS cust_days_since
               FROM h GROUP BY customer_id),
        b AS (SELECT DISTINCT customer_id, t_dat, article_id FROM h),
        p AS (SELECT x.article_id AS item, y.article_id AS partner, COUNT(*) AS n
              FROM b x JOIN b y ON x.customer_id = y.customer_id AND x.t_dat = y.t_dat
                                AND x.article_id <> y.article_id GROUP BY ALL),
        pc AS (SELECT ci.customer_id, p.partner AS article_id, SUM(p.n) AS partner_n
               FROM ci JOIN p ON ci.article_id = p.item GROUP BY ALL)
        SELECT cand.customer_id, cand.article_id,
               COALESCE(ci.n_buy, 0) AS n_buy, COALESCE(ci.days_since, 999) AS days_since,
               COALESCE(it.pop_1w, 0) AS pop_1w, COALESCE(it.pop_2w, 0) AS pop_2w,
               COALESCE(it.pop_8w, 0) AS pop_8w, it.item_price,
               COALESCE(age_i.item_age_w, -1) AS item_age_w,
               COALESCE(pc.partner_n, 0) AS partner_n,
               COALESCE(cu.cust_n, 0) AS cust_n, cu.cust_price,
               COALESCE(cu.cust_days_since, 999) AS cust_days_since,
               c.age
        FROM cand
        LEFT JOIN ci USING (customer_id, article_id)
        LEFT JOIN it USING (article_id)
        LEFT JOIN age_i USING (article_id)
        LEFT JOIN pc USING (customer_id, article_id)
        LEFT JOIN cu USING (customer_id)
        LEFT JOIN customers c USING (customer_id)
    """).df()
    df["price_diff"] = df["item_price"] - df["cust_price"]

    # Text similarity: customer's taste vector vs item vector
    hist = con.execute(f"SELECT DISTINCT customer_id, article_id FROM tx WHERE w BETWEEN {lo} AND {hi}").df()
    grp = hist.groupby("customer_id")["article_id"].apply(list)
    prof = {c: emb[[row_of[a] for a in items]].mean(axis=0) for c, items in grp.items()}
    prof = {c: v / np.linalg.norm(v) for c, v in prof.items()}
    zero = np.zeros(emb.shape[1], dtype=np.float32)
    sim = np.empty(len(df), dtype=np.float32)
    for s in range(0, len(df), 100_000):
        part = df.iloc[s:s + 100_000]
        U = np.stack([prof.get(c, zero) for c in part["customer_id"]])
        I = emb[[row_of[a] for a in part["article_id"]]]
        sim[s:s + len(part)] = (U * I).sum(axis=1)
    df["sim"] = sim

    # Label: 1 if bought in target week t
    tgt = con.execute(f"SELECT DISTINCT customer_id, article_id, 1 AS label FROM tx WHERE w = {t}").df()
    df = df.merge(tgt, on=["customer_id", "article_id"], how="left")
    df["label"] = df["label"].fillna(0).astype(int)
    return df

t0 = time.time()
feat_v = features(cand_v, 1)
print(f"Validation features: {feat_v.shape} | {time.time()-t0:.0f}s")
print(f"Positive rate: {feat_v['label'].mean():.4%}")
print(feat_v.drop(columns=["customer_id", "article_id"]).describe().T[["mean", "min", "max"]].round(3).to_string())

Validation features: (726718, 17) | 123s
Positive rate: 0.4849%
                       mean       min       max
n_buy              0.109525       0.0      25.0
days_since       906.542007       0.0     999.0
pop_1w            51.272939       0.0     137.0
pop_2w           109.375246       0.0     271.0
pop_8w             209.1925       1.0     616.0
item_price          0.03019  0.000791   0.50678
item_age_w        10.956433       0.0      23.0
partner_n            4.0523       0.0     289.0
cust_n            10.072044       0.0     173.0
cust_price         0.028319  0.001062  0.290678
cust_days_since  239.481427       0.0     999.0
age               35.820892      16.0      95.0
price_diff         0.001009 -0.277478  0.449755
sim                 0.45176       0.0       1.0
label              0.004849       0.0       1.0


In [5]:
train_parts = []
for t in [2, 3]:
    t0 = time.time()
    buyers = con.execute(f"SELECT DISTINCT customer_id FROM tx WHERE w = {t}").df()["customer_id"].tolist()
    f = features(candidates(t, buyers), t)
    keep = f.groupby("customer_id")["label"].transform("max") == 1
    f = f[keep]
    train_parts.append(f)
    print(f"week {t}: {len(buyers):,} buyers -> {f['customer_id'].nunique():,} kept | "
          f"{len(f):,} rows | pos rate {f['label'].mean():.2%} | {time.time()-t0:.0f}s")

train = pd.concat(train_parts, ignore_index=True)
print(f"\nTraining rows: {len(train):,} | positives: {train['label'].sum():,}")

week 2: 15,220 buyers -> 3,061 kept | 182,765 rows | pos rate 2.16% | 210s
week 3: 15,930 buyers -> 2,931 kept | 173,215 rows | pos rate 2.10% | 200s

Training rows: 355,980 | positives: 7,594


In [6]:
FEATS = ["n_buy", "days_since", "pop_1w", "pop_2w", "pop_8w", "item_price", "item_age_w",
         "partner_n", "cust_n", "cust_price", "cust_days_since", "age", "price_diff", "sim"]

train = train.sort_values("customer_id", kind="stable").reset_index(drop=True)
fv = feat_v.sort_values("customer_id", kind="stable").reset_index(drop=True)
grp_tr = train.groupby("customer_id", sort=False).size().to_numpy()
grp_v = fv.groupby("customer_id", sort=False).size().to_numpy()

ranker = lgb.LGBMRanker(objective="lambdarank", n_estimators=500, learning_rate=0.05,
                        num_leaves=31, min_child_samples=20, random_state=42, verbose=-1)
t0 = time.time()
ranker.fit(train[FEATS], train["label"], group=grp_tr,
           eval_set=[(fv[FEATS], fv["label"])], eval_group=[grp_v],
           eval_at=[12], eval_metric="map",
           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)])
print(f"Best iteration: {ranker.best_iteration_} | {time.time()-t0:.0f}s")

# Rank candidates and build top-12 lists
top2 = con.execute("""SELECT article_id FROM tx WHERE w BETWEEN 2 AND 3
    GROUP BY article_id ORDER BY COUNT(DISTINCT customer_id) DESC, article_id LIMIT 12""").df()["article_id"].tolist()

def fill(items, filler, k=12):
    out = list(items[:k])
    for f in filler:
        if len(out) >= k: break
        if f not in out: out.append(f)
    return out

fv["score"] = ranker.predict(fv[FEATS])
ranked = fv.sort_values(["customer_id", "score"], ascending=[True, False]).groupby("customer_id").head(12)
lgb_lists = ranked.groupby("customer_id")["article_id"].apply(list).to_dict()

ap_lgb = np.array([apk(actual[c], fill(lgb_lists.get(c, []), top2)) for c in actual])
print(f"LightGBM ranker MAP@12 (validation): {ap_lgb.mean():.5f}")
print("Earlier best: bought-before + bought-together = 0.02554")

C:\Users\RAJA VINAY KUMAR\AppData\Roaming\Python\Python312\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[50]	valid_0's map@12: 0.854605	valid_0's ndcg@12: 0.869405
Best iteration: 26 | 5s
LightGBM ranker MAP@12 (validation): 0.02827
Earlier best: bought-before + bought-together = 0.02554


In [7]:
# Rebuild the best classic method (Step 7) inside this notebook
pdf = con.execute("""
    SELECT customer_id, LIST(article_id ORDER BY last_buy DESC, n DESC, article_id) AS items
    FROM (SELECT customer_id, article_id, MAX(t_dat) AS last_buy, COUNT(*) AS n
          FROM tx WHERE w BETWEEN 2 AND 9 GROUP BY customer_id, article_id)
    GROUP BY customer_id""").df()
past = {c: list(i) for c, i in zip(pdf["customer_id"], pdf["items"])}

prt = con.execute("""
    WITH b AS (SELECT DISTINCT customer_id, t_dat, article_id FROM tx WHERE w BETWEEN 2 AND 9),
    p AS (SELECT x.article_id AS item, y.article_id AS partner, COUNT(*) AS n
          FROM b x JOIN b y ON x.customer_id = y.customer_id AND x.t_dat = y.t_dat
                            AND x.article_id <> y.article_id GROUP BY ALL),
    r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY item ORDER BY n DESC, partner) AS rk FROM p)
    SELECT item, LIST(partner ORDER BY rk) AS partners FROM r WHERE rk <= 3 GROUP BY item""").df()
partners = {i: list(p) for i, p in zip(prt["item"], prt["partners"])}

def with_partners(items):
    out = list(items)
    for it in items:
        for q in partners.get(it, []):
            if q not in out: out.append(q)
    return out

ap_classic = np.array([apk(actual[c], fill(with_partners(past.get(c, [])), top2)) for c in actual])
print(f"Classic best (rebuilt): {ap_classic.mean():.5f}   (Step 7 gave 0.02554)")

def paired_gain(ap_new, ap_old, n_boot=1000, seed=42):
    diff = ap_new - ap_old
    rng = np.random.default_rng(seed)
    boot = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(n_boot)]
    low, high = np.percentile(boot, [2.5, 97.5])
    return diff.mean(), low, high, (diff > 0).sum(), (diff < 0).sum()

gain, low, high, better, worse = paired_gain(ap_lgb, ap_classic)
print(f"\nLightGBM vs classic: gain {gain:+.5f} | 95% range {low:+.5f} to {high:+.5f}")
print(f"Customers better: {better:,}   worse: {worse:,}")
print("Real gain" if low > 0 else "Not clearly real (range includes 0)")

has_hist = np.array([c in past for c in actual])
for name, m in [("With history", has_hist), ("No history", ~has_hist)]:
    print(f"{name:<13} n={m.sum():>6,} | classic {ap_classic[m].mean():.5f} | LightGBM {ap_lgb[m].mean():.5f}")

Classic best (rebuilt): 0.02554   (Step 7 gave 0.02554)

LightGBM vs classic: gain +0.00273 | 95% range +0.00161 to +0.00387
Customers better: 994   worse: 616
Real gain
With history  n= 8,969 | classic 0.03754 | LightGBM 0.04099
No history    n= 5,502 | classic 0.00598 | LightGBM 0.00753


In [8]:
imp = pd.DataFrame({
    "feature": FEATS,
    "gain": ranker.booster_.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
imp["share_%"] = (100 * imp["gain"] / imp["gain"].sum()).round(1)
print(imp[["feature", "share_%"]].to_string(index=False))

        feature  share_%
     days_since     35.1
cust_days_since     14.9
      partner_n     13.2
         pop_1w      8.8
            sim      5.5
         pop_8w      5.3
     price_diff      4.3
     item_age_w      3.8
            age      2.6
         pop_2w      2.4
     item_price      2.0
     cust_price      1.0
         cust_n      0.9
          n_buy      0.3


## LightGBM ranker

This is a common two-stage approach for recommendation: first collect a shortlist of possible items for each customer, then train a model to put them in the right order.


For each customer I collected about 50 candidates from four places: items they bought in the last 8 weeks, the top 3 bought-together partners of those items, the 10 closest items by text embedding, and the 30 best-sellers. The weak spot is right here. Only 7.7% of real purchases ended up in these lists, and only 19% of customers had even one correct item in theirs. No ranking can recover an item that isn't on the list, so that's the ceiling.

I described each customer–item pair with 14 numbers: how recently and how often the customer bought that item, how popular the item was over 1, 2 and 8 weeks, how long it had been on sale, price and the gap to the customer's usual price, bought-together strength, text similarity, and a few customer details. I trained on target weeks 2 and 3, always with features built from older weeks only, and dropped customers whose shortlist had no purchase in it, because a ranker can't learn anything from those.

The ranker reached MAP@12 = 0.02827 on validation, compared with 0.02554 for my best simple method. A paired bootstrap put the gain between +0.00161 and +0.00387, and 994 customers improved while 616 got worse. It helped both groups: +9% for customers with recent history and +26% for those without, where it seems to pick better from the best-sellers by looking at last week's trends.

The most important features were how recently the customer bought that exact item (35%), how recently they shopped at all (15%) and bought-together strength (13%). Text similarity added about 5%. That matches what the simpler methods showed.

Two caveats. Early stopping chose 26 trees using validation, and LightGBM's own MAP counts customers with no correct candidate as perfect, which inflates its internal number. So this validation score is a little optimistic. The test week will be the honest check.